In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import os
import numpy as np

In [ ]:
N = 1000
P = 100

stats_path = os.path.expanduser(f'~/hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/summary_stats/')

results_path = os.path.expanduser(f'~/hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/hpc_pyspoc_scripts/results/concat/')
dataset_folder = f"syn_data_N{N}_P{P}"
dm_file = 'datamatrix__3configs__filtered_st75_dt25.parquet'

data_path = os.path.join(results_path,dataset_folder,dm_file)

In [ ]:
data_matrix = pd.read_parquet(data_path)

In [ ]:
data_matrix_names = data_matrix.columns.str.replace(stats_path, ".", regex=False)
data_matrix.columns = data_matrix_names

In [ ]:
# rem duplicate cols
dup = data_matrix.T.duplicated()
df_dedup = data_matrix.loc[:, ~dup]

# rem constant cols
df_dedup = df_dedup.loc[:,~(df_dedup.nunique() <= 1)]

df_dedup_scaled = (df_dedup - df_dedup.mean())/df_dedup.std()
print(f"dropped {dup.sum()} duplicate columns")

In [ ]:
dup[dup]

In [ ]:
data_matrix.shape, df_dedup.shape

In [ ]:
pyspoc_stat_fam = pd.Series(data_matrix.columns.str.split(".").str[-6:-1].str.join("."), index=data_matrix.columns)

In [ ]:
# pyspoc_stat_fam

In [ ]:
df_module_cluster = df_dedup

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
# load = pd.DataFrame(pca.components_.T, index=df_module_cluster.columns, columns=[f"PC{i}" for i in range(n_pcs)])
# # load
# for pc in load:
#     print(pc, load[pc].abs().sort_values(ascending=False).head(10).index.str.split("statistics.").str[-1].tolist(), "\n")

In [ ]:
def coarse_group(v):
    if v.startswith("GMM_"):
        return "clustered"
    if v.startswith("cpm_counts_"):
        return "count_cpm"
    if v.startswith("counts_"):
        return "count_raw"
    if v.startswith("gp_lmm_"):
        return "manifold"
    if "hypercube_grid" in v:
        return "grid"
    if v.endswith("gaussian"):
        return "gaussian"
    return v

# print(coarse.value_counts())

In [ ]:
coarse.value_counts()

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

X = StandardScaler().fit_transform(df_module_cluster)

n_pcs = 20
pca = PCA(n_components=n_pcs)
Z = pca.fit_transform(X)

groups = df_module_cluster.index.str.split("__").str[0]
grp_keys = {v: coarse_group(v) for v in groups.unique()}
coarse = groups.map(grp_keys)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for g in coarse.unique():
    m = coarse == g
    ax.scatter(Z[m, 0], Z[m, 1], s=12, alpha=0.7, label=g)

ev = pca.explained_variance_ratio_
ax.set_xlabel(f"PC1 ({ev[0]:.0%})"); ax.set_ylabel(f"PC2 ({ev[1]:.0%})")
ax.legend(fontsize=8, markerscale=1.5, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
y, names = pd.factorize(coarse)

In [ ]:
# groups.unique()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

acc = cross_val_score(LogisticRegression(max_iter=5000), X = X, y = y, cv=20)
print(f"accuracy {acc.mean():.2f} +/- {acc.std():.2f}, chance {groups.value_counts(normalize=True).max():.2f}")

In [ ]:
from sklearn.manifold import TSNE

E = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(Z)

fig, ax = plt.subplots(figsize=(4, 3))
for g in sorted(coarse.unique()):
    m = (coarse == g)
    ax.scatter(E[m, 0], E[m, 1], s=10, alpha=0.7, label=g)
ax.set_xticks([]); ax.set_yticks([]); ax.set_title("t-SNE")
ax.legend(fontsize=8, markerscale=1.5, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.clustermap(df_module_cluster.fillna(0), cmap="RdBu_r", center=0, vmin=-3, vmax=3,
               xticklabels=False, yticklabels=False, figsize=(12, 10))

plt.xlabel("statistics"); plt.ylabel("datasets")
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df_module_cluster.fillna(0), cmap="RdBu_r", center=0, vmin=-3, vmax=3,
               xticklabels=False, yticklabels=False)

plt.xlabel("statistics"); plt.ylabel("datasets")
plt.show()

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

In [ ]:
corr = df_module_cluster.corr(method='spearman')
abs_corr = corr.abs()

In [ ]:
dissim_link= linkage(squareform(1-abs_corr,checks=False),method='average')

In [ ]:
# find clusters/modules in the data
modules = pd.Series(fcluster(dissim_link,t=0.2,criterion='distance'), index=abs_corr.index)

In [ ]:
modules.value_counts().value_counts().sort_values(ascending=False)

In [ ]:
print("found " , modules.nunique(), "modules out of ", df_module_cluster.shape[1], " features")
print(f'module id with size..\n{modules.value_counts()[:3]}')
print(f'module size counts\n{modules.value_counts().value_counts()}')

In [ ]:
pyspoc_stat_fam = pd.Series(data_matrix.columns.str.split(".").str[-4:].str.join("."), index=data_matrix.columns)

In [ ]:
for m in modules.value_counts().index[:5]:
    members = modules.index[modules == m]
    print(f"module {m} ({len(members)} features):")
    print(pyspoc_stat_fam[members].value_counts(), "\n")

In [ ]:
# code to look at spearman correlations
# corr = df_dedup.corr(method="spearman")
# upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
# pairs = upper.stack().sort_values(key=abs, ascending=False)
# exact = pairs[pairs.abs() == 1.0]
# print(len(exact), "pairs with corr = 1")
# print(exact.head(30))

In [ ]:
# OLD: CHANGED NOW
# # the data matrix is quite bloated with 'Moment' features right now
# # i think that reducer needs to be changed. im just dropping it for now.

# is_moment_vec = data_matrix.columns.str.contains(r"Moment\.\w+_\d+$", regex=True)
# df_no_moment = data_matrix.loc[:, ~is_moment_vec]
# print(f"dropped {is_moment_vec.sum()} per-column moment features")

In [ ]:
# sp.moment(cols,axis=1,order=[1,2,3]).mean(axis=1)
# n = data.shape[0]
# cols = data[~np.eye(n, dtype=bool)].reshape(n, n - 1)     # row i = column i of data, without its diagonal entry
# import numpy as np
# import scipy.stats as sp

# rng = np.random.default_rng(0)
# data = rng.standard_normal((100, 100))   # stand-in for a 5 x 4 matrix

# print(sp.moment(data, order=[1]))  # default axis=0: one value per column
# print(sp.moment(data, order=[1], axis=None))  # whole matrix: one value
# print(sp.moment(data, order=[1, 2, 3]).mean(axis=1))